# 05 — Feature Engineering v2

## Business Context

To predict which countries a song will chart in next, we need features that capture **why music crosses borders**. Prior research on cross-cultural media diffusion (Hofstede, 2001; Straubhaar, 1991) identifies three key drivers:

1. **Cultural proximity** — Music spreads more easily between culturally similar countries (shared language, similar values, geographic nearness). A hit in Spain is more likely to chart in Mexico than in Japan.
2. **Artist momentum** — Established artists with prior international presence have existing fan bases that accelerate cross-border diffusion.
3. **Market dynamics** — Target country characteristics (chart turnover rate, market size, streaming volume) determine how receptive a market is to new entries.

## Feature Design Principles

- **No data leakage:** Every feature is computed using only information available at the **observation point** (track's first chart date). Artist history, chart ranks, and market statistics are strictly historical.
- **One row per (track, target_country):** The model scores all 61 candidate countries for each track, so features must capture both track-level properties and the track-target relationship.
- **Temporal split:** Train ≤ 2019 | Val 2020 | Test 2021 (by track's first chart date) — ensures the model generalizes to future time periods.

## Feature Categories (113 columns total)

| Category | Count | Description | Business Rationale |
|----------|-------|-------------|--------------------|
| Country rank columns | 62 | Track's top200 rank in each country on observation day | Current geographic footprint — where is the song already popular? |
| Audio features | 12 | All Spotify audio descriptors (danceability, energy, key, loudness, mode, etc.) | Musical characteristics that may appeal differently across cultures |
| Track metadata | 5 | Duration, explicit flag, release timing, viral50 presence | Release strategy signals (Friday releases, viral chart presence) |
| Artist features | 7 | Prior chart count, geographic reach, best rank, collaborations | Artist momentum and international track record |
| Target country features | 11 | Population, streaming volume, chart turnover, continent | Target market size and receptiveness to new entries |
| Origin-target relationship | 6 | Cultural distance, language overlap, song language match, geographic proximity | Core diffusion drivers — cultural corridor + song-language signal |
| Temporal features | 2 | Observation month and year | Seasonal patterns (holiday releases, summer hits) |
| Labels | 2 | did_enter_within_60d, days_to_entry | Binary spread label + regression target |

In [ ]:
## ── Cell 1: Setup & Imports ──────────────────────────────────────────────────

import duckdb
import pathlib
import os

# Paths
ROOT = pathlib.Path(os.getcwd()).parent  # repo root
V2_PATH = ROOT / "datasets" / "v2" / "full"
AUX_PATH = ROOT / "datasets" / "v1_aux"
OUT_PATH = ROOT / "datasets" / "v3_features"
OUT_PATH.mkdir(parents=True, exist_ok=True)

# DuckDB — disk-backed for memory safety with 25M+ rows
DB_FILE = str(OUT_PATH / "_feature_eng.duckdb")
con = duckdb.connect(DB_FILE)
con.execute("SET threads = 4")
con.execute("SET memory_limit = '4GB'")

# Load v2 dataset as a view
con.execute(f"""
    CREATE OR REPLACE VIEW v2 AS
    SELECT * FROM read_parquet('{V2_PATH}/year=*/data_0.parquet', hive_partitioning=true)
""")

# Load auxiliary tables
con.execute(f"""
    CREATE OR REPLACE VIEW cultural_distance AS
    SELECT * FROM read_parquet('{AUX_PATH}/cultural_distance_long.parquet')
""")
con.execute(f"""
    CREATE OR REPLACE VIEW cultural_top5 AS
    SELECT * FROM read_parquet('{AUX_PATH}/cultural_distance_top5.parquet')
""")
con.execute(f"""
    CREATE OR REPLACE VIEW countries_ref AS
    SELECT * FROM read_parquet('{AUX_PATH}/countries_reference_clean.parquet')
""")

# Quick sanity check
for name in ['v2', 'cultural_distance', 'cultural_top5', 'countries_ref']:
    cnt = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name}: {cnt:,} rows")

## 1. Data Foundation

The next three cells build the base tables:
- **Country list** (66 markets) — extracted from the cleaned v2 dataset, excluding Global aggregate and countries removed during data cleaning (South Korea, Russia, Ukraine due to data quality issues)
- **Track-level table** — one row per unique track, joining audio features and metadata from the first chart observation. Audio features (danceability, energy, valence, etc.) are included because prior work on music recommendation suggests they capture genre/mood preferences that vary by culture (Schedl et al., 2018).
- **Observation points** — the track's first chart date becomes the single observation time. This is a critical design choice: using first chart day ensures we only use information available at prediction time (no future leakage).

In [2]:
## ── Cell 2: Country List & Region Normalization ─────────────────────────────

import re

# Extract the 66 distinct countries from v2
countries = con.execute("SELECT DISTINCT region FROM v2 ORDER BY region").fetchdf()['region'].tolist()
print(f"Number of countries: {len(countries)}")

# Snake-case mapping for column names: "United States" → "united_states"
def to_snake(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')

country_to_col = {c: f"rank_{to_snake(c)}" for c in countries}

# Register as a DuckDB table for later joins
con.execute("CREATE OR REPLACE TABLE country_list AS SELECT UNNEST($1::VARCHAR[]) AS country", [countries])

# Show a few examples
for c in countries[:5]:
    print(f"  {c:30s} → {country_to_col[c]}")
print(f"  ... ({len(countries)} total)")

Number of countries: 66
  Andorra                        → rank_andorra
  Argentina                      → rank_argentina
  Australia                      → rank_australia
  Austria                        → rank_austria
  Belgium                        → rank_belgium
  ... (66 total)


In [3]:
## ── Cell 3: Track-Level Table ────────────────────────────────────────────────
# One row per track_id (~195K tracks) with static attributes:
#   first_chart_date, audio features, release_date, explicit, duration_ms, artist
# first_chart_date is based on top200 entries only.

con.execute("""
    CREATE OR REPLACE TABLE tracks AS
    WITH first_appearance AS (
        SELECT
            track_id,
            MIN(date) AS first_chart_date,
            -- Take audio features & metadata from the first chart appearance
            FIRST(artist ORDER BY date, region)        AS artist,
            FIRST(release_date ORDER BY date, region)  AS release_date,
            FIRST(duration_ms ORDER BY date, region)   AS duration_ms,
            FIRST(explicit ORDER BY date, region)       AS explicit,
            FIRST(af_danceability ORDER BY date, region)     AS af_danceability,
            FIRST(af_energy ORDER BY date, region)           AS af_energy,
            FIRST(af_valence ORDER BY date, region)          AS af_valence,
            FIRST(af_tempo ORDER BY date, region)            AS af_tempo,
            FIRST(af_acousticness ORDER BY date, region)     AS af_acousticness,
            FIRST(af_speechiness ORDER BY date, region)      AS af_speechiness,
            FIRST(af_instrumentalness ORDER BY date, region) AS af_instrumentalness,
            FIRST(af_liveness ORDER BY date, region)         AS af_liveness,
            FIRST(af_key ORDER BY date, region)              AS af_key,
            FIRST(af_loudness ORDER BY date, region)         AS af_loudness,
            FIRST(af_mode ORDER BY date, region)             AS af_mode,
            FIRST(af_time_signature ORDER BY date, region)   AS af_time_signature
        FROM v2
        WHERE chart = 'top200'
        GROUP BY track_id
    )
    SELECT
        *,
        -- is_friday_release: release_date falls on a Friday (dayofweek=5 in DuckDB, 0=Mon)
        CASE WHEN dayofweek(release_date) = 4 THEN 1 ELSE 0 END AS is_friday_release
    FROM first_appearance
""")

cnt = con.execute("SELECT COUNT(*) FROM tracks").fetchone()[0]
print(f"Track table: {cnt:,} tracks")
con.execute("SELECT * FROM tracks LIMIT 3").fetchdf()

Track table: 115,473 tracks


,track_id,first_chart_date,artist,release_date,duration_ms,explicit,af_danceability,af_energy,af_valence,af_tempo,af_acousticness,af_speechiness,af_instrumentalness,af_liveness,is_friday_release
0,7vGuf3Y35N4wmASOKLUVVU,2017-08-11,"Marshmello, Khalid",2017-08-11,180822,False,0.520,0.761,0.286,141.971,0.2560,0.0853,0.000005,0.1700,0
1,2z4pcBLQXF2BXKFvd0BuB6,2017-11-10,Jason Derulo,2017-11-10,187521,False,0.845,0.709,0.620,98.062,0.0233,0.0714,0.000000,0.0940,0
2,38yBBH2jacvDxrznF7h08J,2017-10-20,Niall Horan,2017-10-20,188174,False,0.734,0.418,0.868,85.909,0.0129,0.0425,0.000000,0.0579,0


In [4]:
## ── Cell 4: Observation Points ───────────────────────────────────────────────
# Single observation per track: the first day it charts in ANY country.
# Then we look at what happens in the next 60 days.

con.execute("""
    CREATE OR REPLACE TABLE observation_points AS
    SELECT track_id, first_chart_date AS observation_time
    FROM tracks
""")

obs_cnt = con.execute("SELECT COUNT(*) FROM observation_points").fetchone()[0]
print(f"Observation points: {obs_cnt:,} (one per track)")

# Sanity: should equal number of tracks
tracks_cnt = con.execute("SELECT COUNT(*) FROM tracks").fetchone()[0]
assert obs_cnt == tracks_cnt, f"Mismatch: {obs_cnt} obs vs {tracks_cnt} tracks"
print("Confirmed: exactly one observation per track.")

Observation points: 115,473 (one per track)
Confirmed: exactly one observation per track.


## 2. Current Chart Footprint (66 rank columns + viral50 flag)

The most direct predictor of *where* a song will chart next is *where it already charts*. We pivot the top200 chart data to create one column per country, recording the track's rank (1–200) on the observation date, or 0 if not charted.

**Why top200 only (not viral50)?** The viral50 chart captures social sharing momentum but has no stream counts and different ranking mechanics. We use viral50 presence as a single binary flag (`track_in_viral50_at_obs`) rather than separate rank columns, because viral50 ranks don't carry the same ordinal meaning as top200 stream-based ranks.

**Business intuition:** A song ranked #5 in Germany and #150 in Austria has a very different spread profile than one ranked #5 in the US and nowhere else. The rank vector serves as a "geographic fingerprint" of the song's current popularity distribution.

In [5]:
## ── Cell 5: Rank Columns (Pivot) + Viral50 Flag ─────────────────────────────
# For each (track_id, observation_time), pivot top200 chart data into 62 rank columns.
# rank_XX = rank in country XX on the observation_time date (top200 only), or 0 if not charted.
# Also compute track_in_viral50_at_obs: 1 if the track appears on ANY viral50 chart on that day.

# Build the PIVOT SQL dynamically for all 62 countries
pivot_cases = ",\n        ".join([
    f"COALESCE(MAX(CASE WHEN region = '{c}' THEN rank END), 0) AS {country_to_col[c]}"
    for c in countries
])

con.execute(f"""
    CREATE OR REPLACE TABLE obs_with_ranks AS
    WITH top200_ranks AS (
        SELECT
            op.track_id,
            op.observation_time,
            {pivot_cases}
        FROM observation_points op
        LEFT JOIN v2
            ON v2.track_id = op.track_id
            AND v2.date = op.observation_time
            AND v2.chart = 'top200'
        GROUP BY op.track_id, op.observation_time
    ),
    viral_flag AS (
        SELECT
            op.track_id,
            op.observation_time,
            MAX(1) AS track_in_viral50_at_obs
        FROM observation_points op
        JOIN v2
            ON v2.track_id = op.track_id
            AND v2.date = op.observation_time
            AND v2.chart = 'viral50'
        GROUP BY op.track_id, op.observation_time
    )
    SELECT
        t.*,
        COALESCE(v.track_in_viral50_at_obs, 0) AS track_in_viral50_at_obs
    FROM top200_ranks t
    LEFT JOIN viral_flag v
        ON v.track_id = t.track_id AND v.observation_time = t.observation_time
""")

cnt = con.execute("SELECT COUNT(*) FROM obs_with_ranks").fetchone()[0]
cols = con.execute("SELECT COUNT(*) FROM information_schema.columns WHERE table_name = 'obs_with_ranks'").fetchone()[0]
viral_cnt = con.execute("SELECT SUM(track_in_viral50_at_obs) FROM obs_with_ranks").fetchone()[0]
print(f"obs_with_ranks: {cnt:,} rows × {cols} columns (2 ids + {cols - 3} rank cols + viral50 flag)")
print(f"Tracks on viral50 at observation time: {viral_cnt:,} ({viral_cnt/cnt*100:.1f}%)")

# Spot check: show a row with some non-zero ranks
con.execute("""
    SELECT * FROM obs_with_ranks
    WHERE rank_united_states > 0 AND rank_brazil > 0
    LIMIT 1
""").fetchdf()

obs_with_ranks: 115,473 rows × 69 columns (2 ids + 66 rank cols + viral50 flag)
Tracks on viral50 at observation time: 8,254 (7.1%)


,track_id,observation_time,rank_andorra,rank_argentina,rank_australia,rank_austria,rank_belgium,rank_bolivia,rank_brazil,rank_bulgaria,...,rank_switzerland,rank_taiwan,rank_thailand,rank_turkey,rank_united_arab_emirates,rank_united_kingdom,rank_united_states,rank_uruguay,rank_vietnam,track_in_viral50_at_obs
0,56lBX2WHqfGuoeZxZvUnVj,2017-05-01,0,0,197,70,149,0,157,0,...,87,109,0,121,0,127,118,0,0,1


## 3. Artist Features (7 columns)

An artist's international track record is a strong prior for whether a new release will spread. We compute 7 features using **only chart data before the observation date** (no leakage):

| Feature | Description | Business Rationale |
|---------|-------------|-------------------|
| `artist_prior_chart_count` | Total chart appearances before observation | Overall visibility and catalog depth |
| `artist_prior_unique_regions` | Distinct countries charted in before | Geographic reach — artists with broad reach spread faster |
| `artist_prior_best_rank` | Best rank ever achieved (201 if none) | Peak popularity signal |
| `artist_prior_unique_tracks` | Distinct prior charted tracks | Prolificness vs one-hit-wonder distinction |
| `multi_artist_flag` | 1 if collaboration (feat., &, x) | Collaborations pool fan bases across markets |
| `artist_country_ratio` | unique_regions / unique_tracks | Internationalization rate per release |
| `artist_prior_success_in_target` | Prior tracks by this artist in the target country | Direct evidence of existing fan base in the target market |

**Why `artist_prior_success_in_target` is target-specific:** This is the only artist feature that varies per (track, country) pair. If Drake has had 15 prior hits in Canada but only 2 in Japan, the model should treat these target countries differently — this feature captures that asymmetry.

In [6]:
## ── Cell 6: Artist Features ──────────────────────────────────────────────────
# All computed using only top200 data with date < observation_time to prevent leakage.
# 7 features: prior_chart_count, prior_unique_regions, prior_best_rank,
#              prior_unique_tracks, multi_artist_flag, country_ratio,
#              prior_success_in_target (added later in cross-join cell)

# First, compute the multi_artist_flag from the artist string (static per track)
con.execute("""
    CREATE OR REPLACE TABLE track_artist_flag AS
    SELECT
        track_id,
        artist,
        CASE WHEN artist LIKE '%feat.%'
             OR artist LIKE '%Feat.%'
             OR artist LIKE '%, %'
             OR artist LIKE '% & %'
             OR artist LIKE '% x %'
             OR artist LIKE '% X %'
        THEN 1 ELSE 0 END AS multi_artist_flag
    FROM tracks
""")

# Artist-level prior features at each observation_time (top200 only)
con.execute("""
    CREATE OR REPLACE TABLE artist_features AS
    WITH prior_data AS (
        -- All top200 chart entries by the same artist BEFORE the observation_time
        SELECT
            op.track_id AS obs_track_id,
            op.observation_time,
            t.artist,
            v2.track_id AS prior_track_id,
            v2.region,
            v2.rank AS chart_rank,
            v2.date
        FROM observation_points op
        JOIN tracks t ON op.track_id = t.track_id
        JOIN v2 ON v2.artist = t.artist
              AND v2.date < op.observation_time
              AND v2.chart = 'top200'
    ),
    agg AS (
        SELECT
            obs_track_id AS track_id,
            observation_time,
            COUNT(*) AS artist_prior_chart_count,
            COUNT(DISTINCT region) AS artist_prior_unique_regions,
            MIN(chart_rank) AS artist_prior_best_rank,
            COUNT(DISTINCT prior_track_id) AS artist_prior_unique_tracks
        FROM prior_data
        GROUP BY obs_track_id, observation_time
    )
    SELECT
        a.track_id,
        a.observation_time,
        COALESCE(a.artist_prior_chart_count, 0) AS artist_prior_chart_count,
        COALESCE(a.artist_prior_unique_regions, 0) AS artist_prior_unique_regions,
        COALESCE(a.artist_prior_best_rank, 201) AS artist_prior_best_rank,
        COALESCE(a.artist_prior_unique_tracks, 0) AS artist_prior_unique_tracks,
        -- country_ratio = unique_regions / unique_tracks (0 if no prior tracks)
        CASE WHEN COALESCE(a.artist_prior_unique_tracks, 0) = 0 THEN 0.0
             ELSE a.artist_prior_unique_regions * 1.0 / a.artist_prior_unique_tracks
        END AS artist_country_ratio
    FROM agg a
""")

cnt = con.execute("SELECT COUNT(*) FROM artist_features").fetchone()[0]
print(f"Artist features computed for {cnt:,} (track, observation_time) pairs")
con.execute("SELECT * FROM artist_features LIMIT 3").fetchdf()

Artist features computed for 71,766 (track, observation_time) pairs


,track_id,observation_time,artist_prior_chart_count,artist_prior_unique_regions,artist_prior_best_rank,artist_prior_unique_tracks,artist_country_ratio
0,0woHVOh3KXLSDv8VU9XwZ4,2018-07-03,4819,18,1,1,18.000000
1,03uDtNwgN3S9tGqL2ojnXH,2018-07-06,693,1,23,5,0.200000
2,37zuIvk4KBkAxxLJsxJaHq,2019-03-03,3216,36,7,39,0.923077


## 4. Target Cross-Join and Target Country Features (11 columns)

For each track, we create rows for all 65 candidate countries (excluding the track's origin countries). The cross-join produces the (track, target_country) pairs the model will score.

**Target country features** capture market-level characteristics:

| Feature | Description | Why It Matters |
|---------|-------------|----------------|
| `target_population` | Country population | Larger markets have more chart turnover and entry opportunities |
| `target_avg_daily_streams` | Mean daily top200 streams over 30 days before observation | Market streaming activity level — proxy for market size and engagement |
| `target_new_entry_rate_30d` | Rate of new chart entries in target over 30 days | Chart dynamism — markets with high turnover are easier to enter |
| `target_continent_*` | One-hot encoded continent (6 dummies) | Regional clustering — captures geographic/cultural macro-patterns |

**Why a 30-day window for market statistics?** The 30-day window captures *current* market conditions while smoothing out daily noise. Shorter windows (7 days) are too volatile; longer windows (90+ days) miss seasonal shifts. The 30-day window is a standard choice in time-series feature engineering for capturing recent trends.

In [7]:
## ── Cell 7: Target Cross-Join ────────────────────────────────────────────────
# For each (track_id, observation_time), cross-join with all 62 countries.
# Exclude countries where the track is already charting on top200 (rank > 0).
# Also compute artist_prior_success_in_target here (target-specific, top200 only).

# Build the condition to check if rank > 0 in any country
rank_cols = list(country_to_col.values())

# We need a reverse mapping: column name → country name
col_to_country = {v: k for k, v in country_to_col.items()}

# Build CASE expression: for each target country, check if its rank column > 0
filter_cases = " OR ".join([
    f"(cl.country = '{c}' AND owr.{country_to_col[c]} > 0)"
    for c in countries
])

con.execute(f"""
    CREATE OR REPLACE TABLE pairs AS
    SELECT
        owr.*,
        cl.country AS target_country
    FROM obs_with_ranks owr
    CROSS JOIN country_list cl
    -- Exclude countries where track is already charting on top200
    WHERE NOT ({filter_cases})
""")

cnt = con.execute("SELECT COUNT(*) FROM pairs").fetchone()[0]
print(f"Pair-level rows (after excluding already-charting): {cnt:,}")

# Now add artist_prior_success_in_target (top200 only)
con.execute("""
    CREATE OR REPLACE TABLE artist_target_success AS
    SELECT
        op.track_id,
        op.observation_time,
        v2.region AS target_country,
        COUNT(DISTINCT v2.track_id) AS artist_prior_success_in_target
    FROM observation_points op
    JOIN tracks t ON op.track_id = t.track_id
    JOIN v2 ON v2.artist = t.artist
              AND v2.date < op.observation_time
              AND v2.track_id != op.track_id
              AND v2.chart = 'top200'
    GROUP BY op.track_id, op.observation_time, v2.region
""")

cnt2 = con.execute("SELECT COUNT(*) FROM artist_target_success").fetchone()[0]
print(f"Artist-target success entries: {cnt2:,}")

Pair-level rows (after excluding already-charting): 7,357,050
Artist-target success entries: 593,066


In [8]:
## ── Cell 8: Target Country Features ──────────────────────────────────────────
# Join country metadata (population, continent) from countries_reference_clean.
# Compute target_avg_daily_streams and target_new_entry_rate_30d from trailing 30-day windows.
# Uses top200 data only (streams are NULL for viral50 anyway).

# Optimize for laptop runs: compute reusable (date, country) rolling stats once,
# then expand back to the required (track, observation_time, target_country) grain.
con.execute("""
    CREATE OR REPLACE TABLE target_country_stats AS
    WITH observation_calendar AS (
        SELECT observation_time
        FROM generate_series(
            (SELECT MIN(observation_time) FROM observation_points),
            (SELECT MAX(observation_time) FROM observation_points),
            INTERVAL '1 day'
        ) AS t(observation_time)
    ),
    country_day_stats AS (
        SELECT
            region AS target_country,
            date AS stat_date,
            SUM(streams) AS total_streams,
            COUNT(*) AS chart_entries,
            SUM(CASE WHEN trend = 'NEW_ENTRY' THEN 1 ELSE 0 END) AS new_entries
        FROM v2
        WHERE chart = 'top200'
        GROUP BY region, date
    ),
    calendar_country_stats AS (
        SELECT
            oc.observation_time,
            cl.country AS target_country,
            COALESCE(cds.total_streams, 0) AS total_streams,
            COALESCE(cds.chart_entries, 0) AS chart_entries,
            COALESCE(cds.new_entries, 0) AS new_entries
        FROM observation_calendar oc
        CROSS JOIN country_list cl
        LEFT JOIN country_day_stats cds
               ON cds.target_country = cl.country
              AND cds.stat_date = oc.observation_time
    ),
    country_window_stats AS (
        SELECT
            observation_time,
            target_country,
            SUM(total_streams) OVER w * 1.0 / NULLIF(SUM(chart_entries) OVER w, 0) AS target_avg_daily_streams,
            SUM(new_entries) OVER w * 1.0 / NULLIF(SUM(chart_entries) OVER w, 0) AS target_new_entry_rate_30d
        FROM calendar_country_stats
        WINDOW w AS (
            PARTITION BY target_country
            ORDER BY observation_time
            ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING
        )
    )
    SELECT
        op.track_id,
        op.observation_time,
        cws.target_country,
        cws.target_avg_daily_streams,
        cws.target_new_entry_rate_30d
    FROM observation_points op
    JOIN country_window_stats cws
      ON cws.observation_time = op.observation_time
""")

cnt = con.execute("SELECT COUNT(*) FROM target_country_stats").fetchone()[0]
print(f"Target country stats: {cnt:,} track-country entries")

# Show sample
con.execute("""
    SELECT target_country, 
           ROUND(AVG(target_avg_daily_streams), 0) AS avg_streams,
           ROUND(AVG(target_new_entry_rate_30d), 4) AS avg_new_entry_rate
    FROM target_country_stats 
    GROUP BY target_country 
    ORDER BY avg_streams DESC 
    LIMIT 10
""").fetchdf()

Target country stats: 7,621,218 track-country entries


,target_country,avg_streams,avg_new_entry_rate
0,United States,388399.0,0.0545
1,Brazil,154804.0,0.0548
2,Mexico,130480.0,0.0485
3,Germany,112975.0,0.0615
4,United Kingdom,100379.0,0.0752
5,Spain,77065.0,0.0455
6,Italy,73098.0,0.0485
7,France,67916.0,0.0561
8,Australia,55888.0,0.0536
9,Argentina,54279.0,0.0496


## 5. Origin-Target Relationship Features (6 columns)

These are the **core diffusion features** — they capture the cultural and geographic relationship between where the song *already charts* (origin set) and each *target country*. These features are grounded in Hofstede's cultural dimensions theory (2001) and the "cultural proximity" framework from international media studies (Straubhaar, 1991).

| Feature | Description | Academic Rationale |
|---------|-------------|-------------------|
| `cultural_dist_min` | Minimum Hofstede distance from any origin country to target | Lower distance = more similar cultures = easier diffusion. Uses minimum (not mean) because a song only needs *one* culturally close origin to spread. |
| `cultural_dist_missing` | 1 if cultural distance data is unavailable | 19.4% of country pairs lack Hofstede data (non-Western nations). This flag lets the model learn a separate pattern for under-represented markets rather than relying on imputed values. |
| `same_language_flag` | 1 if any origin country shares an official language with target | **Country-language proxy.** Captures multi-market language overlap — whether *any* charting country shares a language with the target. Effective because it encodes diffusion context across the full origin set, but imperfect as a proxy for song language (see below). |
| `song_lang_matches_target` | 1 if the LLM-detected song language matches the target country's primary language | **Direct song-language signal.** Captures the listener-side language barrier — does the song's actual language match what the target audience speaks? Complements `same_language_flag` by providing ground-truth song language. |
| `same_continent_flag` | 1 if any origin country is on the same continent | Geographic proximity correlates with shared media ecosystems, time zones, and marketing reach. |
| `neighbor_entered_count` | Count of target's top-5 cultural neighbors already in origin set | Captures "cultural corridor" effects — if 3 of Brazil's 5 nearest cultural neighbors already chart the song, Brazil is likely next. |

**Why Hofstede over simpler geography?** Pure geographic distance misses cultural connections (e.g., UK–Australia are geographically distant but culturally close). Hofstede's 6-dimension framework (power distance, individualism, masculinity, uncertainty avoidance, long-term orientation, indulgence) provides a validated, multi-dimensional measure of cultural similarity. The 19.4% missingness is handled transparently via the flag column and median imputation (median cultural distance = 1.620).

---

### Why two language features?

Language is arguably the single most important cultural barrier in music consumption. A listener is far more likely to engage with a song they can understand or that matches the linguistic conventions of their market. We capture this with **two complementary features**:

**`same_language_flag` (country-language proxy):** This flag checks whether *any* country the track is already charting in shares an official language with the target. It is a proxy because it uses country languages, not the language the song is actually sung in. This creates systematic mismatches — Swedish, Norwegian, Dutch, and Polish artists frequently release English-language tracks (e.g., Avicii, Kygo, Tiësto), but the proxy assigns their national language instead. Similarly, cross-border first charts (e.g., a Brazilian song first charting in Turkey) and multi-language markets (Belgium, Switzerland, Morocco) produce inaccurate mappings. Despite this, the proxy remains valuable because it captures a multi-market diffusion signal: which language communities the song has *already penetrated*.

**`song_lang_matches_target` (LLM-detected song language):** To address the proxy's limitations, we detect the actual song language using a local LLM (qwen3.5:4b via Ollama). Traditional NLP-based language detection (e.g., `langdetect`, `fasttext`) fails on short song titles because character n-gram statistics need substantial text — a 2-word title is insufficient. The LLM overcomes this by leveraging *world knowledge*: given a title, artist name, and origin country, it reasons about the artist's nationality and typical singing language. For example, "Borderline" by Szpaku is correctly identified as Polish because the model knows Szpaku is a Polish rapper — even though the title is an English word. We use an assistant-prefill trick to bypass the model's chain-of-thought overhead, achieving ~0.7s/title.

**Why both, not just one?** The two features capture different signals. `song_lang_matches_target` directly answers "does the listener understand this song?" — a first-order language barrier. `same_language_flag` answers "has this song already entered a market that speaks the target's language?" — a second-order diffusion signal. A Spanish song that has already charted in Spain (same_language_flag=1 for Mexico) is a stronger predictor for Mexico than a Spanish song that has only charted in Germany, even though both have song_lang_matches_target=1 for Mexico. The model benefits from having both.

In [9]:
## ── Cell 9: Origin↔Target Relationship Features ─────────────────────────────
# 5 features: cultural_dist_min, cultural_dist_missing_flag,
#              same_language_flag, same_continent_flag, neighbor_entered_count

# We need to know which countries the track is currently charting in (origin set).
# This is encoded in the rank columns: rank > 0 means the track is in that country.

# Step 1: Build an "origin set" table — (track_id, observation_time, origin_country)
origin_unions = " UNION ALL ".join([
    f"SELECT track_id, observation_time, '{c}' AS origin_country FROM obs_with_ranks WHERE {country_to_col[c]} > 0"
    for c in countries
])

con.execute(f"""
    CREATE OR REPLACE TABLE origin_set AS
    {origin_unions}
""")

print(f"Origin set rows: {con.execute('SELECT COUNT(*) FROM origin_set').fetchone()[0]:,}")

# Step 2: cultural_dist_min — MIN distance from any origin country to the target
con.execute("""
    CREATE OR REPLACE TABLE cultural_dist_features AS
    SELECT
        p.track_id,
        p.observation_time,
        p.target_country,
        MIN(cd.cultural_distance) AS cultural_dist_min,
        CASE WHEN MIN(cd.cultural_distance) IS NULL THEN 1 ELSE 0 END AS cultural_dist_missing
    FROM pairs p
    JOIN origin_set os
        ON os.track_id = p.track_id AND os.observation_time = p.observation_time
    LEFT JOIN cultural_distance cd
        ON cd.source_country = os.origin_country AND cd.target_country = p.target_country
    GROUP BY p.track_id, p.observation_time, p.target_country
""")

# Impute missing cultural distances with the global median
median_dist = con.execute("""
    SELECT MEDIAN(cultural_dist_min) FROM cultural_dist_features WHERE cultural_dist_min IS NOT NULL
""").fetchone()[0]
print(f"Median cultural distance (for imputation): {median_dist:.3f}")

con.execute(f"""
    UPDATE cultural_dist_features
    SET cultural_dist_min = {median_dist}
    WHERE cultural_dist_min IS NULL
""")

# Step 3: same_language_flag — any origin country shares language with target
con.execute("""
    CREATE OR REPLACE TABLE language_features AS
    SELECT
        p.track_id,
        p.observation_time,
        p.target_country,
        MAX(CASE
            WHEN cr_origin.official_language IS NOT NULL
             AND cr_target.official_language IS NOT NULL
             AND (
                -- Check if any word in origin language appears in target language
                -- Simple approach: check if the primary language matches
                SPLIT_PART(cr_origin.official_language, ',', 1) = SPLIT_PART(cr_target.official_language, ',', 1)
                OR cr_origin.official_language LIKE '%' || SPLIT_PART(cr_target.official_language, ',', 1) || '%'
                OR cr_target.official_language LIKE '%' || SPLIT_PART(cr_origin.official_language, ',', 1) || '%'
             )
            THEN 1 ELSE 0
        END) AS same_language_flag,
        MAX(CASE
            WHEN cr_origin.continent = cr_target.continent THEN 1 ELSE 0
        END) AS same_continent_flag
    FROM pairs p
    JOIN origin_set os
        ON os.track_id = p.track_id AND os.observation_time = p.observation_time
    LEFT JOIN countries_ref cr_origin ON cr_origin.country = os.origin_country
    LEFT JOIN countries_ref cr_target ON cr_target.country = p.target_country
    GROUP BY p.track_id, p.observation_time, p.target_country
""")

# Step 4: neighbor_entered_count — of target's top-5 cultural neighbors,
#          how many has the track already entered?
con.execute("""
    CREATE OR REPLACE TABLE neighbor_features AS
    SELECT
        p.track_id,
        p.observation_time,
        p.target_country,
        COALESCE(SUM(CASE WHEN os.origin_country IS NOT NULL THEN 1 ELSE 0 END), 0)
            AS neighbor_entered_count
    FROM pairs p
    LEFT JOIN cultural_top5 ct5
        ON ct5.source_country = p.target_country
    LEFT JOIN origin_set os
        ON os.track_id = p.track_id
        AND os.observation_time = p.observation_time
        AND os.origin_country = ct5.target_country
    GROUP BY p.track_id, p.observation_time, p.target_country
""")

for tbl in ['cultural_dist_features', 'language_features', 'neighbor_features']:
    cnt = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"{tbl}: {cnt:,} rows")

Origin set rows: 264,168
Median cultural distance (for imputation): 1.620
cultural_dist_features: 7,357,050 rows
language_features: 7,357,050 rows
neighbor_features: 7,357,050 rows


In [ ]:
## ── Cell 9b: LLM-Based Song Language Detection ──────────────────────────────
# Detect the actual language each song is sung in using a local LLM (qwen3.5:4b).
# This produces `song_lang_matches_target`: 1 if the detected song language
# matches the target country's primary language.
#
# Requirements: Ollama running locally with qwen3.5:4b pulled.
#   ollama pull qwen3.5:4b
#   ollama serve

import requests, time, os, json as _json

OLLAMA_MODEL = "qwen3.5:4b"
OLLAMA_URL = "http://localhost:11434/api/chat"
CACHE_PATH = os.path.join(ROOT, "datasets", "processed", "song_language_cache.json")

SYSTEM_PROMPT = (
    "Detect the language each song is sung in. "
    "Use the artist origin country AND the title language as clues. "
    "Most artists sing in the language of their origin country. "
    "Reply with ONLY the ISO 639-1 two-letter code. Nothing else."
)

# Country name → primary language ISO 639-1 code
COUNTRY_PRIMARY_LANG = {
    'Sweden': 'sv', 'Norway': 'no', 'Denmark': 'da', 'Finland': 'fi', 'Iceland': 'is',
    'United States': 'en', 'United Kingdom': 'en', 'Ireland': 'en', 'Australia': 'en',
    'New Zealand': 'en', 'Canada': 'en', 'South Africa': 'en', 'Singapore': 'en',
    'Germany': 'de', 'Austria': 'de', 'Switzerland': 'de', 'Luxembourg': 'de',
    'France': 'fr', 'Belgium': 'fr',
    'Spain': 'es', 'Mexico': 'es', 'Argentina': 'es', 'Chile': 'es', 'Colombia': 'es',
    'Peru': 'es', 'Ecuador': 'es', 'Costa Rica': 'es', 'Dominican Republic': 'es',
    'Guatemala': 'es', 'Honduras': 'es', 'Nicaragua': 'es', 'Panama': 'es',
    'Paraguay': 'es', 'El Salvador': 'es', 'Uruguay': 'es', 'Bolivia': 'es', 'Venezuela': 'es',
    'Brazil': 'pt', 'Portugal': 'pt',
    'Italy': 'it', 'Netherlands': 'nl', 'Poland': 'pl', 'Czech Republic': 'cs', 'Slovakia': 'sk',
    'Hungary': 'hu', 'Romania': 'ro', 'Bulgaria': 'bg', 'Croatia': 'hr',
    'Turkey': 'tr', 'Israel': 'he', 'Egypt': 'ar', 'Morocco': 'ar',
    'Saudi Arabia': 'ar', 'United Arab Emirates': 'ar',
    'Japan': 'ja', 'Taiwan': 'zh', 'Hong Kong': 'zh',
    'India': 'hi', 'Indonesia': 'id', 'Malaysia': 'ms', 'Philippines': 'tl',
    'Thailand': 'th', 'Vietnam': 'vi',
    'Estonia': 'et', 'Latvia': 'lv', 'Lithuania': 'lt', 'Greece': 'el', 'Cyprus': 'el',
    'Andorra': 'ca',
}

# ── Step 1: Get unique tracks with their origin country ──────────────────────
tracks_for_llm = con.execute("""
    SELECT DISTINCT t.track_id, t.title, t.artist,
           FIRST(fc.region) AS origin_country
    FROM tracks t
    JOIN (
        SELECT track_id, region,
               ROW_NUMBER() OVER (PARTITION BY track_id ORDER BY date) AS rn
        FROM v2 WHERE chart = 'top200'
    ) fc ON fc.track_id = t.track_id AND fc.rn = 1
    GROUP BY t.track_id, t.title, t.artist
""").fetchdf()

print(f"Unique tracks to process: {len(tracks_for_llm):,}")

# ── Step 2: Load cache (skip already-detected tracks) ────────────────────────
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        lang_cache = _json.load(f)
    print(f"Loaded {len(lang_cache):,} cached detections from {CACHE_PATH}")
else:
    lang_cache = {}

to_detect = tracks_for_llm[~tracks_for_llm['track_id'].isin(lang_cache)]
print(f"Remaining to detect: {len(to_detect):,}")

# ── Step 3: Run LLM detection (one title at a time, with prefill trick) ──────
if len(to_detect) > 0:
    t0 = time.time()
    done = 0
    save_every = 500  # checkpoint cache every 500 tracks

    for _, row in to_detect.iterrows():
        title = row['title']
        artist = row['artist']
        origin = row['origin_country']

        try:
            resp = requests.post(OLLAMA_URL, json={
                "model": OLLAMA_MODEL,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f'"{title}" by {artist} (from {origin})'},
                    {"role": "assistant", "content": "The ISO 639-1 code is: "},
                ],
                "stream": False,
                "options": {"temperature": 0, "num_predict": 16},
            }, timeout=30)
            raw = resp.json()["message"]["content"].strip().lower().replace('*', '')
            pred = None
            for tok in raw.replace('.', ' ').replace(',', ' ').split():
                if len(tok) == 2 and tok.isalpha():
                    pred = tok
                    break
            lang_cache[row['track_id']] = pred
        except Exception as e:
            lang_cache[row['track_id']] = None

        done += 1
        if done % save_every == 0:
            with open(CACHE_PATH, 'w') as f:
                _json.dump(lang_cache, f)
            elapsed = time.time() - t0
            rate = done / elapsed
            remaining = (len(to_detect) - done) / rate / 3600
            print(f"  {done:,}/{len(to_detect):,} detected "
                  f"({elapsed/60:.0f}min elapsed, ~{remaining:.1f}h remaining)")

    # Final save
    with open(CACHE_PATH, 'w') as f:
        _json.dump(lang_cache, f)
    elapsed = time.time() - t0
    print(f"\nDetection complete: {done:,} tracks in {elapsed/60:.1f} min "
          f"({elapsed/done*1000:.0f}ms/title)")
else:
    print("All tracks already cached — skipping LLM inference.")

# ── Step 4: Build song_lang_matches_target table ─────────────────────────────
# Map each track to its detected language, then compare with target country language
tracks_for_llm['detected_lang'] = tracks_for_llm['track_id'].map(lang_cache)

detected_count = tracks_for_llm['detected_lang'].notna().sum()
print(f"\nDetected language for {detected_count:,}/{len(tracks_for_llm):,} tracks "
      f"({detected_count/len(tracks_for_llm)*100:.1f}%)")
print(f"Top detected languages:\n{tracks_for_llm['detected_lang'].value_counts().head(10)}")

# Register as DuckDB table and build the match feature
con.execute("DROP TABLE IF EXISTS song_lang_detected")
con.register('_song_lang_df', tracks_for_llm[['track_id', 'detected_lang']])
con.execute("""
    CREATE TABLE song_lang_detected AS
    SELECT track_id, detected_lang FROM _song_lang_df
""")

# Build country → primary language mapping in DuckDB
country_lang_rows = [(k, v) for k, v in COUNTRY_PRIMARY_LANG.items()]
con.execute("DROP TABLE IF EXISTS country_primary_lang")
con.execute("CREATE TABLE country_primary_lang (country VARCHAR, primary_lang VARCHAR)")
con.executemany("INSERT INTO country_primary_lang VALUES (?, ?)", country_lang_rows)

con.execute("""
    CREATE OR REPLACE TABLE song_lang_match_features AS
    SELECT
        p.track_id,
        p.observation_time,
        p.target_country,
        CASE
            WHEN sld.detected_lang IS NOT NULL
             AND cpl.primary_lang IS NOT NULL
             AND sld.detected_lang = cpl.primary_lang
            THEN 1 ELSE 0
        END AS song_lang_matches_target
    FROM pairs p
    LEFT JOIN song_lang_detected sld ON sld.track_id = p.track_id
    LEFT JOIN country_primary_lang cpl ON cpl.country = p.target_country
""")

match_stats = con.execute("""
    SELECT
        COUNT(*) AS total,
        SUM(song_lang_matches_target) AS matches,
        ROUND(AVG(song_lang_matches_target) * 100, 1) AS match_pct
    FROM song_lang_match_features
""").fetchone()
print(f"\nsong_lang_matches_target: {match_stats[1]:,} / {match_stats[0]:,} = {match_stats[2]}%")

## 6. Labels

Two prediction targets are computed for each (track, target_country) pair:

**`did_enter_within_60d`** (binary, {0, 1}):
- **1** if the track's *first* top200 appearance in the target country occurs within 60 days *after* the observation date (strictly after, to prevent leakage from same-day entries that are already captured in the rank columns)
- **0** otherwise (never enters target's chart, or enters after 60 days)
- **Class imbalance:** ~0.9% positive rate in the full dataset (ratio ~110:1). This extreme imbalance reflects reality — most songs chart in only a handful of countries. The model must learn to identify the rare spreading events.

**`days_to_entry`** (integer, [1, 60]):
- For positive rows: number of days from observation date to first chart entry in target country
- For negative rows: NULL (undefined — the song never entered)
- Used as the regression target for the auxiliary days-to-entry model

In [10]:
## ── Cell 10: Labels ──────────────────────────────────────────────────────────
# did_enter_within_60d: 1 if track enters top200 in target within 60 days of observation_time
# days_to_entry: days until first top200 entry (NULL for negatives)

# First top200 chart date per (track_id, country)
con.execute("""
    CREATE OR REPLACE TABLE track_country_first_chart AS
    SELECT track_id, region AS country, MIN(date) AS first_chart_date
    FROM v2
    WHERE chart = 'top200'
    GROUP BY track_id, region
""")

con.execute("""
    CREATE OR REPLACE TABLE labels AS
    SELECT
        p.track_id,
        p.observation_time,
        p.target_country,
        CASE
            WHEN tcfc.first_chart_date IS NOT NULL
             AND tcfc.first_chart_date > p.observation_time
             AND tcfc.first_chart_date <= p.observation_time + INTERVAL '60 days'
            THEN 1 ELSE 0
        END AS did_enter_within_60d,
        CASE
            WHEN tcfc.first_chart_date IS NOT NULL
             AND tcfc.first_chart_date > p.observation_time
             AND tcfc.first_chart_date <= p.observation_time + INTERVAL '60 days'
            THEN (tcfc.first_chart_date - p.observation_time)::INTEGER
            ELSE NULL
        END AS days_to_entry
    FROM pairs p
    LEFT JOIN track_country_first_chart tcfc
        ON tcfc.track_id = p.track_id AND tcfc.country = p.target_country
""")

pos = con.execute("SELECT SUM(did_enter_within_60d) FROM labels").fetchone()[0]
total = con.execute("SELECT COUNT(*) FROM labels").fetchone()[0]
print(f"Labels: {total:,} rows, {pos:,} positives ({pos/total*100:.2f}%)")
print(f"Class ratio (neg:pos): {(total-pos)/pos:.1f}:1")

# days_to_entry distribution for positives
con.execute("""
    SELECT
        MIN(days_to_entry) AS min_days,
        ROUND(AVG(days_to_entry), 1) AS avg_days,
        MEDIAN(days_to_entry) AS median_days,
        MAX(days_to_entry) AS max_days
    FROM labels
    WHERE did_enter_within_60d = 1
""").fetchdf()

Labels: 7,357,050 rows, 66,299 positives (0.90%)
Class ratio (neg:pos): 110.0:1


,min_days,avg_days,median_days,max_days
0,1,12.0,4.0,60


## 7. Temporal Features, Final Assembly, and Downsampling

**Temporal features** (`observation_month`, `observation_year`) capture seasonal effects: holiday releases in December, summer hits in June–August, and year-over-year growth in streaming adoption. While the temporal split prevents the model from memorizing specific years, the month feature allows it to learn recurring seasonal patterns.

### Training Set Downsampling

The natural positive rate (~0.9%) creates an extreme class imbalance that slows training and wastes compute on uninformative negative examples. We apply **5:1 negative downsampling** to the training set only:

- **Training set:** Downsampled to ~16.7% positive rate (5 negatives per positive). This accelerates gradient boosting training while preserving enough negative examples for the model to learn meaningful decision boundaries.
- **Validation and test sets:** Kept at the natural class ratio (~0.6–0.7% positive). This is critical — evaluation must reflect the real-world distribution to produce unbiased metrics.
- **Reproducibility:** Downsampling uses `TABLESAMPLE ... (bernoulli, 42)` with a fixed seed.

**Why 5:1 and not 1:1?** Extreme undersampling (1:1) discards too many informative negatives and can distort learned boundaries. The 5:1 ratio is a common heuristic in imbalanced learning that balances training efficiency against information retention (He & Garcia, 2009).

In [11]:
## ── Cell 11: Temporal Features & Final Assembly ──────────────────────────────
# Combine ALL features into the final table.
# Add observation_month, observation_year, days_since_release, is_friday_release.
# One-hot encode target_country_continent.
# Include track_in_viral50_at_obs flag from Cell 5.

# Build the continent one-hot columns dynamically
continents = con.execute("""
    SELECT DISTINCT continent FROM countries_ref WHERE continent IS NOT NULL ORDER BY continent
""").fetchdf()['continent'].tolist()
print(f"Continents for one-hot: {continents}")

continent_cases = ",\n        ".join([
    f"CASE WHEN cr.continent = '{cont}' THEN 1 ELSE 0 END AS target_continent_{to_snake(cont)}"
    for cont in continents
])

# Build rank column list for SELECT
rank_col_list = ", ".join([f"p.{col}" for col in rank_cols])

con.execute(f"""
    CREATE OR REPLACE TABLE features_final AS
    SELECT
        -- Identifiers
        p.track_id,
        p.observation_time,
        p.target_country,

        -- A. Per-country rank columns (66, top200 only)
        {rank_col_list},

        -- B. Audio features (12)
        t.af_danceability,
        t.af_energy,
        t.af_valence,
        t.af_tempo,
        t.af_acousticness,
        t.af_speechiness,
        t.af_instrumentalness,
        t.af_liveness,
        t.af_key,
        t.af_loudness,
        t.af_mode,
        t.af_time_signature,

        -- C. Track metadata (5)
        t.duration_ms,
        CASE WHEN t.explicit THEN 1 ELSE 0 END AS explicit,
        CASE WHEN t.release_date IS NOT NULL
             THEN (p.observation_time - t.release_date)::INTEGER
             ELSE NULL
        END AS days_since_release,
        COALESCE(t.is_friday_release, 0) AS is_friday_release,
        p.track_in_viral50_at_obs,

        -- D. Artist features (7)
        COALESCE(af.artist_prior_chart_count, 0)    AS artist_prior_chart_count,
        COALESCE(af.artist_prior_unique_regions, 0)  AS artist_prior_unique_regions,
        COALESCE(af.artist_prior_best_rank, 201)     AS artist_prior_best_rank,
        COALESCE(af.artist_prior_unique_tracks, 0)   AS artist_prior_unique_tracks,
        COALESCE(taf.multi_artist_flag, 0)           AS multi_artist_flag,
        COALESCE(af.artist_country_ratio, 0.0)       AS artist_country_ratio,
        COALESCE(ats.artist_prior_success_in_target, 0) AS artist_prior_success_in_target,

        -- E. Target country features (4 + continent one-hot)
        COALESCE(cr.population, 0)                   AS target_population,
        COALESCE(tcs.target_avg_daily_streams, 0)    AS target_avg_daily_streams,
        COALESCE(tcs.target_new_entry_rate_30d, 0)   AS target_new_entry_rate_30d,
        {continent_cases},

        -- F. Origin↔Target relationship features (6)
        COALESCE(cdf.cultural_dist_min, {median_dist})  AS cultural_dist_min,
        COALESCE(cdf.cultural_dist_missing, 1)          AS cultural_dist_missing,
        COALESCE(lf.same_language_flag, 0)              AS same_language_flag,
        COALESCE(slm.song_lang_matches_target, 0)       AS song_lang_matches_target,
        COALESCE(lf.same_continent_flag, 0)             AS same_continent_flag,
        COALESCE(nf.neighbor_entered_count, 0)          AS neighbor_entered_count,

        -- G. Temporal features (2)
        EXTRACT(MONTH FROM p.observation_time)::INTEGER AS observation_month,
        EXTRACT(YEAR FROM p.observation_time)::INTEGER  AS observation_year,

        -- Targets
        lb.did_enter_within_60d,
        lb.days_to_entry

    FROM pairs p
    JOIN tracks t ON t.track_id = p.track_id
    LEFT JOIN artist_features af
        ON af.track_id = p.track_id AND af.observation_time = p.observation_time
    LEFT JOIN track_artist_flag taf ON taf.track_id = p.track_id
    LEFT JOIN artist_target_success ats
        ON ats.track_id = p.track_id AND ats.observation_time = p.observation_time
        AND ats.target_country = p.target_country
    LEFT JOIN target_country_stats tcs
        ON tcs.track_id = p.track_id AND tcs.observation_time = p.observation_time
        AND tcs.target_country = p.target_country
    LEFT JOIN countries_ref cr ON cr.country = p.target_country
    LEFT JOIN cultural_dist_features cdf
        ON cdf.track_id = p.track_id AND cdf.observation_time = p.observation_time
        AND cdf.target_country = p.target_country
    LEFT JOIN language_features lf
        ON lf.track_id = p.track_id AND lf.observation_time = p.observation_time
        AND lf.target_country = p.target_country
    LEFT JOIN neighbor_features nf
        ON nf.track_id = p.track_id AND nf.observation_time = p.observation_time
        AND nf.target_country = p.target_country
    LEFT JOIN song_lang_match_features slm
        ON slm.track_id = p.track_id AND slm.observation_time = p.observation_time
        AND slm.target_country = p.target_country
    LEFT JOIN labels lb
        ON lb.track_id = p.track_id AND lb.observation_time = p.observation_time
        AND lb.target_country = p.target_country
""")

total = con.execute("SELECT COUNT(*) FROM features_final").fetchone()[0]
ncols = con.execute("SELECT COUNT(*) FROM information_schema.columns WHERE table_name = 'features_final'").fetchone()[0]
print(f"\nFinal feature table: {total:,} rows × {ncols} columns")

# Show column names
cols_df = con.execute("SELECT column_name, data_type FROM information_schema.columns WHERE table_name = 'features_final' ORDER BY ordinal_position").fetchdf()
print(f"\nAll columns ({len(cols_df)}):")
for _, row in cols_df.iterrows():
    print(f"  {row['column_name']:45s} {row['data_type']}")

Continents for one-hot: ['Africa', 'Antarctica', 'Asia', 'Europe', 'North America', 'Oceania', 'South America']

Final feature table: 7,357,050 rows × 108 columns

All columns (108):
  track_id                                      VARCHAR
  observation_time                              DATE
  target_country                                VARCHAR
  rank_andorra                                  INTEGER
  rank_argentina                                INTEGER
  rank_australia                                INTEGER
  rank_austria                                  INTEGER
  rank_belgium                                  INTEGER
  rank_bolivia                                  INTEGER
  rank_brazil                                   INTEGER
  rank_bulgaria                                 INTEGER
  rank_canada                                   INTEGER
  rank_chile                                    INTEGER
  rank_colombia                                 INTEGER
  rank_costa_rica                   

In [12]:
## ── Cell 12: Class Balance & Downsampling ────────────────────────────────────
# Report positive/negative ratio.
# Downsample negatives to 5:1 ratio within training set only (applied in Cell 13).

pos_rate = con.execute("""
    SELECT
        SUM(did_enter_within_60d) AS positives,
        COUNT(*) - SUM(did_enter_within_60d) AS negatives,
        COUNT(*) AS total,
        ROUND(SUM(did_enter_within_60d) * 100.0 / COUNT(*), 2) AS pos_pct,
        ROUND((COUNT(*) - SUM(did_enter_within_60d)) * 1.0 / NULLIF(SUM(did_enter_within_60d), 0), 1) AS neg_to_pos_ratio
    FROM features_final
""").fetchdf()
print("Class balance (full dataset):")
print(pos_rate.to_string(index=False))

# Breakdown by year
print("\nClass balance by observation_year:")
con.execute("""
    SELECT
        observation_year,
        SUM(did_enter_within_60d) AS positives,
        COUNT(*) AS total,
        ROUND(SUM(did_enter_within_60d) * 100.0 / COUNT(*), 2) AS pos_pct
    FROM features_final
    GROUP BY observation_year
    ORDER BY observation_year
""").fetchdf()

Class balance (full dataset):
 positives  negatives   total  pos_pct  neg_to_pos_ratio
   66299.0  7290751.0 7357050      0.9             110.0

Class balance by observation_year:


,observation_year,positives,total,pos_pct
0,2017,18618.0,1350036,1.38
1,2018,14387.0,1342473,1.07
2,2019,12508.0,1481453,0.84
3,2020,10419.0,1647221,0.63
4,2021,10367.0,1535867,0.67


## 8. Train/Val/Test Split and Export

The temporal split boundaries:
- **Train (≤ 2019):** 3 years of historical data for model training. Downsampled to 272,922 rows (16.7% positive rate).
- **Validation (2020):** 1 full year for hyperparameter tuning and model selection. 1,647,221 rows at natural rate.
- **Test (2021):** 1 full year for final evaluation. 1,535,867 rows at natural rate.

**Leakage verification:** We explicitly check that no `track_id` appears in more than one split. Since the split is by the track's first chart date (not by individual chart observations), a track can only belong to one split.

**Export format:** Parquet with zstd compression and Hive partitioning by year, consistent with the upstream data pipeline. Each split is exported as a single file for efficient loading during model training.

In [13]:
## ── Cell 13: Train/Val/Test Split ────────────────────────────────────────────
# Train: first_chart_date ≤ 2019-12-31
# Val:   first_chart_date in 2020
# Test:  first_chart_date in 2021
# All rows of same track stay in same split (trivially true: one obs per track).
# Downsample negatives to 5:1 in training set only.

# Add first_chart_date to features_final for splitting
con.execute("""
    CREATE OR REPLACE TABLE features_with_split AS
    SELECT
        ff.*,
        t.first_chart_date,
        CASE
            WHEN t.first_chart_date <= '2019-12-31' THEN 'train'
            WHEN t.first_chart_date >= '2020-01-01' AND t.first_chart_date <= '2020-12-31' THEN 'val'
            WHEN t.first_chart_date >= '2021-01-01' THEN 'test'
        END AS split
    FROM features_final ff
    JOIN tracks t ON t.track_id = ff.track_id
""")

# Split stats before downsampling
print("Split stats (before downsampling):")
con.execute("""
    SELECT
        split,
        COUNT(*) AS rows,
        SUM(did_enter_within_60d) AS positives,
        ROUND(SUM(did_enter_within_60d) * 100.0 / COUNT(*), 2) AS pos_pct,
        COUNT(DISTINCT track_id) AS tracks
    FROM features_with_split
    GROUP BY split
    ORDER BY split
""").fetchdf()

# Downsample training negatives to 5:1 ratio
train_pos = con.execute("SELECT SUM(did_enter_within_60d) FROM features_with_split WHERE split = 'train'").fetchone()[0]
target_neg = train_pos * 5
train_neg_total = con.execute("SELECT COUNT(*) FROM features_with_split WHERE split = 'train' AND did_enter_within_60d = 0").fetchone()[0]

sample_frac = min(1.0, target_neg / train_neg_total) if train_neg_total > 0 else 1.0
print(f"\nTraining set: {train_pos:,} positives, {train_neg_total:,} negatives")
print(f"Downsampling negatives to {target_neg:,} (fraction: {sample_frac:.4f})")

# Use TABLESAMPLE with a seed for reproducibility
con.execute(f"""
    CREATE OR REPLACE TABLE train_set AS
    -- All positives
    SELECT * EXCLUDE (first_chart_date, split) FROM features_with_split
    WHERE split = 'train' AND did_enter_within_60d = 1
    UNION ALL
    -- Downsampled negatives
    SELECT * EXCLUDE (first_chart_date, split) FROM features_with_split
    WHERE split = 'train' AND did_enter_within_60d = 0
    USING SAMPLE {sample_frac * 100:.2f} PERCENT (bernoulli, 42)
""")

con.execute("""
    CREATE OR REPLACE TABLE val_set AS
    SELECT * EXCLUDE (first_chart_date, split) FROM features_with_split WHERE split = 'val'
""")

con.execute("""
    CREATE OR REPLACE TABLE test_set AS
    SELECT * EXCLUDE (first_chart_date, split) FROM features_with_split WHERE split = 'test'
""")

# Final split stats
for name in ['train_set', 'val_set', 'test_set']:
    total = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    pos = con.execute(f"SELECT SUM(did_enter_within_60d) FROM {name}").fetchone()[0]
    tracks = con.execute(f"SELECT COUNT(DISTINCT track_id) FROM {name}").fetchone()[0]
    print(f"\n{name}: {total:,} rows, {pos:,} positives ({pos/total*100:.2f}%), {tracks:,} tracks")

# Verify no track leakage between splits
leakage = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT track_id FROM train_set
        INTERSECT
        SELECT DISTINCT track_id FROM val_set
    )
""").fetchone()[0]
leakage2 = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT track_id FROM train_set
        INTERSECT
        SELECT DISTINCT track_id FROM test_set
    )
""").fetchone()[0]
leakage3 = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT track_id FROM val_set
        INTERSECT
        SELECT DISTINCT track_id FROM test_set
    )
""").fetchone()[0]
print(f"\nTrack leakage check: train∩val={leakage}, train∩test={leakage2}, val∩test={leakage3}")
assert leakage == 0 and leakage2 == 0 and leakage3 == 0, "LEAKAGE DETECTED!"
print("No track leakage between splits.")

Split stats (before downsampling):

Training set: 45,513 positives, 4,128,449 negatives
Downsampling negatives to 227,565 (fraction: 0.0551)

train_set: 272,922 rows, 45,513 positives (16.68%), 64,093 tracks

val_set: 1,647,221 rows, 10,419 positives (0.63%), 25,858 tracks

test_set: 1,535,867 rows, 10,367 positives (0.67%), 24,047 tracks

Track leakage check: train∩val=0, train∩test=0, val∩test=0
No track leakage between splits.


In [14]:
## ── Cell 14: Export ──────────────────────────────────────────────────────────
# Export train/val/test as separate parquet files.
# Log stats: row counts, positive rates, feature distributions.

import json
from datetime import datetime

export_path = OUT_PATH
print(f"Exporting to: {export_path}")

for split_name, table_name in [('train', 'train_set'), ('val', 'val_set'), ('test', 'test_set')]:
    out_file = export_path / f"{split_name}.parquet"
    con.execute(f"""
        COPY (SELECT * FROM {table_name})
        TO '{out_file}' (FORMAT PARQUET, COMPRESSION 'zstd')
    """)
    size_mb = out_file.stat().st_size / 1024 / 1024
    cnt = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"  {split_name}.parquet: {cnt:,} rows, {size_mb:.1f} MB")

# Also export the full (non-downsampled) dataset for reference
full_file = export_path / "full.parquet"
con.execute(f"""
    COPY (SELECT * EXCLUDE (first_chart_date, split) FROM features_with_split)
    TO '{full_file}' (FORMAT PARQUET, COMPRESSION 'zstd')
""")
full_size = full_file.stat().st_size / 1024 / 1024
full_cnt = con.execute("SELECT COUNT(*) FROM features_with_split").fetchone()[0]
print(f"  full.parquet: {full_cnt:,} rows, {full_size:.1f} MB")

# Write manifest
manifest = {
    "created": datetime.now().isoformat(),
    "description": "Feature-engineered dataset for Top-5 Country Ranker (60-day horizon, first-chart-day observation)",
    "splits": {},
}
for split_name, table_name in [('train', 'train_set'), ('val', 'val_set'), ('test', 'test_set')]:
    cnt = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    pos = con.execute(f"SELECT SUM(did_enter_within_60d) FROM {table_name}").fetchone()[0]
    tracks = con.execute(f"SELECT COUNT(DISTINCT track_id) FROM {table_name}").fetchone()[0]
    manifest["splits"][split_name] = {
        "rows": cnt,
        "positives": pos,
        "positive_rate": round(pos / cnt * 100, 2),
        "tracks": tracks,
        "file": f"{split_name}.parquet",
    }

manifest_file = export_path / "manifest.json"
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"\nManifest written to {manifest_file}")
print(json.dumps(manifest, indent=2))

Exporting to: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/processed/v3_features
  train.parquet: 272,922 rows, 13.3 MB
  val.parquet: 1,647,221 rows, 48.3 MB
  test.parquet: 1,535,867 rows, 44.3 MB
  full.parquet: 7,357,050 rows, 239.6 MB

Manifest written to /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/processed/v3_features/manifest.json
{
  "created": "2026-03-14T11:40:37.085910",
  "description": "Feature-engineered dataset for Top-5 Country Ranker (60-day horizon, first-chart-day observation)",
  "splits": {
    "train": {
      "rows": 272922,
      "positives": 45513,
      "positive_rate": 16.68,
      "tracks": 64093,
      "file": "train.parquet"
    },
    "val": {
      "rows": 1647221,
      "positives": 10419,
      "positive_rate": 0.63,
      "tracks": 25858,
      "file": "val.parquet"
    },
    "test": {
      "rows": 1535867,
      "positives": 10367,
      "positive_rate": 0.67,
     

In [15]:
## ── Cell 15: Verification ────────────────────────────────────────────────────
# 1. Schema check, value ranges, no NaN in critical columns
# 2. Verify temporal split correctness (no future leakage)
# 3. Spot-check rows

print("=" * 70)
print("VERIFICATION SUITE")
print("=" * 70)

# 1. Schema check — verify expected column count
expected_feature_cols = (
    62  # rank columns (top200)
    + 12  # audio
    + 5  # track metadata (duration, explicit, days_since_release, is_friday_release, viral50 flag)
    + 7  # artist
    + 3 + len(continents)  # target country (pop, streams, entry_rate + continent one-hots)
    + 6  # relationship (incl. song_lang_matches_target)
    + 2  # temporal
)
expected_total = 3 + expected_feature_cols + 2  # 3 identifiers + features + 2 targets
actual_cols = con.execute("SELECT COUNT(*) FROM information_schema.columns WHERE table_name = 'train_set'").fetchone()[0]
print(f"\n[1] Column count: expected ~{expected_total}, got {actual_cols}")

# 2. Value range checks
print("\n[2] Value range checks:")
checks = [
    ("Rank columns in [0, 200]",
     f"SELECT COUNT(*) FROM train_set WHERE {' OR '.join([f'{col} < 0 OR {col} > 200' for col in rank_cols[:10]])}"),
    ("Audio features in [0, 1] (danceability, energy — others have wider ranges)",
     "SELECT COUNT(*) FROM train_set WHERE af_danceability < 0 OR af_danceability > 1 OR af_energy < 0 OR af_energy > 1"),
    ("did_enter_within_60d in {0, 1}",
     "SELECT COUNT(*) FROM train_set WHERE did_enter_within_60d NOT IN (0, 1)"),
    ("days_to_entry in [1, 60] for positives",
     "SELECT COUNT(*) FROM train_set WHERE did_enter_within_60d = 1 AND (days_to_entry < 1 OR days_to_entry > 60)"),
    ("observation_month in [1, 12]",
     "SELECT COUNT(*) FROM train_set WHERE observation_month < 1 OR observation_month > 12"),
    ("track_in_viral50_at_obs in {0, 1}",
     "SELECT COUNT(*) FROM train_set WHERE track_in_viral50_at_obs NOT IN (0, 1)"),
]
all_passed = True
for desc, query in checks:
    bad = con.execute(query).fetchone()[0]
    status = "PASS" if bad == 0 else f"FAIL ({bad:,} violations)"
    if bad > 0:
        all_passed = False
    print(f"  {status}: {desc}")

# 3. No NaN/NULL in critical columns
print("\n[3] NULL checks on critical columns:")
critical_cols = ['track_id', 'observation_time', 'target_country', 'did_enter_within_60d',
                 'af_danceability', 'af_energy', 'cultural_dist_min', 'track_in_viral50_at_obs']
for col in critical_cols:
    nulls = con.execute(f"SELECT COUNT(*) FROM train_set WHERE {col} IS NULL").fetchone()[0]
    status = "PASS" if nulls == 0 else f"FAIL ({nulls:,} NULLs)"
    if nulls > 0:
        all_passed = False
    print(f"  {status}: {col}")

# 4. Temporal split — no future leakage
print("\n[4] Temporal split verification:")
for split_name, table_name in [('train', 'train_set'), ('val', 'val_set'), ('test', 'test_set')]:
    date_range = con.execute(f"""
        SELECT MIN(t.first_chart_date), MAX(t.first_chart_date)
        FROM {table_name} s JOIN tracks t ON t.track_id = s.track_id
    """).fetchone()
    print(f"  {split_name}: first_chart_date range [{date_range[0]} — {date_range[1]}]")

# 5. Spot-check: sample 3 rows from train set
print("\n[5] Sample rows (train set):")
sample = con.execute("""
    SELECT track_id, observation_time, target_country,
           track_in_viral50_at_obs, artist_prior_chart_count, cultural_dist_min,
           same_language_flag, neighbor_entered_count, did_enter_within_60d, days_to_entry
    FROM train_set
    WHERE did_enter_within_60d = 1
    LIMIT 3
""").fetchdf()
print(sample.to_string(index=False))

# 6. Feature distributions summary
print("\n[6] Feature distributions (train set):")
con.execute("""
    SELECT
        ROUND(AVG(cultural_dist_min), 3) AS avg_cult_dist,
        ROUND(AVG(same_language_flag), 3) AS avg_same_lang,
        ROUND(AVG(same_continent_flag), 3) AS avg_same_cont,
        ROUND(AVG(neighbor_entered_count), 3) AS avg_neighbor_cnt,
        ROUND(AVG(artist_prior_chart_count), 0) AS avg_artist_charts,
        ROUND(AVG(artist_prior_success_in_target), 2) AS avg_artist_target_success,
        ROUND(AVG(track_in_viral50_at_obs), 3) AS viral50_rate,
        ROUND(AVG(days_since_release), 0) AS avg_days_since_rel
    FROM train_set
""").fetchdf()

print("\n" + "=" * 70)
print(f"ALL CHECKS {'PASSED' if all_passed else 'SOME FAILED — review above'}")
print("=" * 70)

# Cleanup: close the DuckDB connection
# (the .duckdb file persists for re-runs; parquet exports are the deliverables)
con.close()
print("\nDuckDB connection closed. Feature engineering complete.")

VERIFICATION SUITE

[1] Column count: expected ~108, got 108

[2] Value range checks:
  PASS: Rank columns in [0, 200]
  PASS: Audio features in [0, 1] (except tempo)
  PASS: did_enter_within_60d in {0, 1}
  PASS: days_to_entry in [1, 60] for positives
  PASS: observation_month in [1, 12]
  PASS: track_in_viral50_at_obs in {0, 1}

[3] NULL checks on critical columns:
  PASS: track_id
  PASS: observation_time
  PASS: target_country
  PASS: did_enter_within_60d
  PASS: af_danceability
  PASS: af_energy
  PASS: cultural_dist_min
  PASS: track_in_viral50_at_obs

[4] Temporal split verification:
  train: first_chart_date range [2017-01-01 — 2019-12-31]
  val: first_chart_date range [2020-01-01 — 2020-12-31]
  test: first_chart_date range [2021-01-01 — 2021-12-31]

[5] Sample rows (train set):
              track_id observation_time target_country  track_in_viral50_at_obs  artist_prior_chart_count  cultural_dist_min  same_language_flag  neighbor_entered_count  did_enter_within_60d  days_to_e